In [ ]:
from langchain_openai import ChatOpenAI
from pydantic import BaseModel,Field
from langchain_core.messages import HumanMessage,AIMessage,ToolMessage,RemoveMessage,BaseMessage
from typing import TypedDict,List,Annotated
from langgraph.graph import StateGraph,add_messages,MessagesState,END,START
from Prompts import Supervisor_prompt_template
import uuid
from Agents.Query_Agent.query_agent import ready_rag_graph
from Agents.Knowledgebase_Agent.knowledge_agent import knowledge_compiled_graph
from Agents.Complaint_Agent.complaint_agent import complaint_compiled_graph
from Agents.Request_Agent.request_agent import compiled_request_graph
from Agents.Communication_Agent.communication_agent import compiled_communication_graph

from Agents.Query_Agent.utils.rag_state import BASE_DIR
from Agents.Query_Agent.utils.internal_rag_pipeline import discover_collections
from langgraph.types import Send,Command
from langgraph.checkpoint.memory import InMemorySaver
llm = ChatOpenAI(model = 'gpt-4o-mini',api_key="  ")
checkpointer = InMemorySaver()


✅ Connection successful!
✅ Query executed successfully. Rows fetched: 1


In [2]:
#Output Schema
class output_state(MessagesState):
    tool_call : bool
    input :str
    tool_calls : List
    output : Annotated[List[BaseMessage],add_messages]
     

class Action_Schema(TypedDict):
    Agent_Name: str = Field(default=None, description='Name of the agent to be invoked')
    message : str = Field(default=None, description='data to be passed to agent')

class output_schema(TypedDict):
    output : str = Field(default = None, description = 'message to respond to user, default value if tool call is required')
    action : List[Action_Schema] = Field(default = None, description = 'list of dictionary with key value pair of Agent_Name and message')

In [ ]:

def supervisor_node(state: output_state):

    print(f"\n\nSUPERVISOR NODE: {state}\n\n")

    last_message = state['messages'][-1]


    # Normal flow
    response_model = llm.with_structured_output(output_schema).invoke([
        HumanMessage(
            content=Supervisor_prompt_template.format(
                chat_history=state['messages'][:-1],
                message=last_message,
                available_collections = discover_collections(BASE_DIR)
                
            )
        )
    ])

    # for i in state['messages']:
    #     print(f"\n**MESSAGE** : {i}\n")
    print((f"\n**MESSAGE** : {response_model}\n"))

    if response_model['action']:
        # Chat-only → update state
        print('**AGENT CHAT**')

        return {'messages':AIMessage(content=f'TOOL CALL ISSUED {response_model['action']}'),'tool_calls' : response_model['action']}
        
    else:
        # Action-only → route to specific agent
        print('**NORMAL CHAT**')

        print('model response : ',response_model['output'])

        return {'messages' : AIMessage(content=response_model['output']) }
        

# Example Nodes
def Complaint_Node(state: dict) -> ToolMessage:
    print(f"\n\nCOMPLAINT NODE: {state}\n\n")
    print(f'COMPLAIN INPUT : {state['input']}')
    complaint_input = state['input']
    
    output = complaint_compiled_graph.invoke({'input': state['input']})

    return {'output':[ToolMessage(content=output['output'],tool_call_id = uuid.uuid4())]}

def Query_Node(state: dict) -> ToolMessage:
    print(f"\n\nQUERY NODE: {state}\n\n")

    rag_input = state['input']

    output = ready_rag_graph.invoke({'input': rag_input})

    return {'output':[ToolMessage(content=output['output'],tool_call_id = uuid.uuid4())]}

def Request_Node(state: dict) -> ToolMessage:
    print(f"\n\nREQUEST NODE : {state}\n\n")

    request_input = state['input']

    output = compiled_request_graph.invoke({'input': request_input,
              'email_id' : None,
              'active_state': None ,
              "final_output" : None,
              "interrupt_question": None})

    return {'output':[ToolMessage(content=output['output'],tool_call_id = uuid.uuid4())]}

def Communication_Node(state: dict) -> ToolMessage:
    print(f"\n\nREQUEST NODE : {state}\n\n")

    communication_input = state['input']

    output = compiled_communication_graph.invoke({'input': communication_input,
              'email_id' : None,
              'active_state': None ,
              "final_output" : None,
              "interrupt_question": None})

    return {'output':[ToolMessage(content=output['output'],tool_call_id = uuid.uuid4())]}

def Knowledge_Node(state: dict) -> ToolMessage:
    print(f"\n\nKNOWLEDGE NODE :{state}\n\n")

    knowledge_input = state['input']

    output = knowledge_compiled_graph.invoke({'input': knowledge_input})

    return {'output':[ToolMessage(content=output['output'],tool_call_id = uuid.uuid4())]}


def Collector_Node(state: dict) -> dict:
    print(f"\n\nCOLLECTOR NODE :{state}\n\n")
    """
    Collect all ToolMessages from state['output'], merge their content,
    and append as a single AIMessage to state['output'].
    Original ToolMessages are removed to avoid duplicates.
    """
    output_list = state.get('output', [])
    tool_messages = [msg for msg in output_list if isinstance(msg, ToolMessage)]

    if not tool_messages:
        return state  # nothing to merge

    remove_message = [RemoveMessage(id = m.id) for m in state['output']]


    # Merge content of all ToolMessages
    merged_content = "\n".join(msg.content for msg in tool_messages)

    return {'messages' : ToolMessage(content=merged_content,tool_call_id = uuid.uuid4()),'output' :remove_message,'tool_calls' : []}


def condition(state : output_state) -> output_state:
    print(f"\n\nCOLLECTOR NODE :{state['tool_calls']}\n\n")
 
    if len(state['tool_calls']) >0:
        print('**AGENT CHAT**')

        state['messages'] = AIMessage(content="Invoking Agent to Complete Tasks")
        sends = [
            Send(resp['Agent_Name'], {'input': resp['message']})
            for resp in state['tool_calls']
        ]
        return sends
    else:
        print('sending to end')
        return 'end'

def end_node(state: output_state) -> output_state:
    return state

In [ ]:
supervsior = StateGraph(output_state)

supervsior.add_node('supervisor',supervisor_node)
supervsior.add_node('Complaint_Agent',Complaint_Node)
supervsior.add_node('Query_Agent',Query_Node)
supervsior.add_node('Document_Download_Agent',Request_Node)
supervsior.add_node('Data_Agent',Knowledge_Node)
supervsior.add_node('Communication_Agent',Communication_Node)
supervsior.add_node('Collector_Node',Collector_Node)
supervsior.add_node('end',end_node)

supervsior.add_edge(START,'supervisor')
supervsior.add_conditional_edges('supervisor',condition,['end','Communication_Agent','Complaint_Agent','Query_Agent','Document_Download_Agent','Data_Agent'])

supervsior.add_edge('Complaint_Agent','Collector_Node')
supervsior.add_edge('Communication_Agent','Collector_Node')
supervsior.add_edge('Query_Agent','Collector_Node')
supervsior.add_edge('Document_Download_Agent','Collector_Node')
supervsior.add_edge('Data_Agent','Collector_Node')
supervsior.add_edge('Collector_Node','supervisor')
supervsior.add_edge('end',END)

compiled_supervisor = supervsior.compile(checkpointer=checkpointer)

In [5]:
config = {'configurable': {'thread_id' : '1'}}

# user_input = 'download civil report for application id QAWS23EDFR45 and i am frustrated as i havent recieved my document yet'
# user_input = 'download cibil for application id qaws23edfr45 and also vm summary for application 2qaswed34rfg'
# user_input = 'what is frontend development'
# user_input = 'give count of open and close cases from database'
# user_input = 'what counts you retrieve'
# user_input = 'i am no happy with portal performance it is always lagging'

# user_input = 'i want to download my document for application id QAWSEDREWQAS'

# input_data = {'messages' : [HumanMessage(content = user_input)],'tool_call': False,'tool_calls':[]}

# supervisor_response = compiled_supervisor.invoke(input_data,config = config)

In [9]:
# user_input = 'i am frustrated as i havent recieved my cibil report yet'
# user_input = 'download me BIRTH CERTIFICATE and send A Confirmation to abhirajsingh.rajpurohit@idolizesolutions.com'
# user_input = 'download cibil for application id qaws23edfr45 and also vm summary for application 2qaswed34rfg'
# user_input = 'what is frontend development'
# user_input = 'what are count of open and close cases from database'
# user_input = 'what counts you retrieve'
# user_input = 'i am no happy with portal performance it is always lagging'
# user_input = 'i want to download my document for application id QAWSEDREWQAS and send mail on abhirajsingh.rajpurohit@idolizesolutions.com'
# user_input = 'vm summary is my document name'
# user_input = 'QAWS23EDFR45'
# user_input = 'is there any data present related to abhiraj search in query agent'


# user_input = 'hyy'
# user_input = 'i am abhiraj singh'
# user_input = 'what is my name'
# user_input = 'what are the available services?'
# user_input = 'what is document download'
# user_input = 'who is abhiraj and what are count of open and close cases from database'
# user_input = 'who are main components of agentic ai'
# user_input = "give my PAN CARD AND AADHAR CARD FOR APPLICATION ID QAWS23EDFR45 and notify me on mail about it  "
user_input = 'go for aadhar card'

In [10]:


input_data = {'messages' : [HumanMessage(content = user_input)],'tool_call': False,'tool_calls':[]}

result = compiled_supervisor.invoke(input_data, config = config)

print(f"\n\n{result}\n\n")

print(result.get('__interrupt__'))
try:
    if result.get('__interrupt__'):

        while True:

            print(f"user input : {result.get('__interrupt__')[0].value}")

            user_query = result.get('__interrupt__')[0].value

            print('an interrupt took place')

            data = input(f"{user_query}")

            result = compiled_supervisor.invoke(
                    Command(resume=data),
                    config=config
                    )
except Exception as e:
    print(e)

for i in result['messages']:
    print(f'\nCHAT : {i}')



SUPERVISOR NODE: {'messages': [HumanMessage(content='give my PAN CARD AND AADHAR CARD FOR APPLICATION ID QAWS23EDFR45 and notify me on mail about it  ', additional_kwargs={}, response_metadata={}, id='37b3d6d5-3fea-4734-bdd5-0560b00e1569'), AIMessage(content='TOOL CALL ISSUED [{\'Agent_Name\': \'Document_Download_Agent\', \'message\': "Download document for document \'PAN CARD\' and Application_id \'QAWS23EDFR45\', \'notify me on mail about it\'"}]', additional_kwargs={}, response_metadata={}, id='f612594e-7251-419b-95aa-b9f99886e4b1'), ToolMessage(content="There was some internal error while downloading the document 'PAN_Card' for Application ID QAWS23EDFR45. Please try again later., email with subject 'Document Download Notification for Application ID QAWS23EDFR45' and body 'Dear Abhiraj Singh,\n\nI hope this message finds you well. I am writing to inform you that there was an internal error while attempting to download the document 'PAN CARD' for Application ID QAWS23EDFR45. Pleas

In [11]:
result

{'messages': [HumanMessage(content='give my PAN CARD AND AADHAR CARD FOR APPLICATION ID QAWS23EDFR45 and notify me on mail about it  ', additional_kwargs={}, response_metadata={}, id='37b3d6d5-3fea-4734-bdd5-0560b00e1569'),
  AIMessage(content='TOOL CALL ISSUED [{\'Agent_Name\': \'Document_Download_Agent\', \'message\': "Download document for document \'PAN CARD\' and Application_id \'QAWS23EDFR45\', \'notify me on mail about it\'"}]', additional_kwargs={}, response_metadata={}, id='f612594e-7251-419b-95aa-b9f99886e4b1'),
  ToolMessage(content="There was some internal error while downloading the document 'PAN_Card' for Application ID QAWS23EDFR45. Please try again later., email with subject 'Document Download Notification for Application ID QAWS23EDFR45' and body 'Dear Abhiraj Singh,\n\nI hope this message finds you well. I am writing to inform you that there was an internal error while attempting to download the document 'PAN CARD' for Application ID QAWS23EDFR45. Please try again lat